# **Explore Hessen Soccer**

In [1]:
from utils import write_scraped_df, concat_scrape, manual_geocode
import pandas as pd
import osmnx as ox
pd.set_option('display.max_rows', 200)

# define coordinate reference system
CRS = 5243

See ``wiki_scraper`` from utils to see how each table was scraped.

In [2]:
liga_dict = {
'verbandsliga_nord': 'https://www.fussball.de/spieltag/vl-grnord-hessen-verbandsliga-herren-saison2627-hessen/-/staffel/0318HV9REO00000AVS5489BUVV628VP4-G#!/section/matches',
'verbandsliga_mitte': 'https://www.fussball.de/spieltag/vl-grmitte-hessen-verbandsliga-herren-saison2627-hessen/-/staffel/0318I3K77C00000BVS5489BUVV628VP4-G#!/section/matches',
'verbandsliga_sued': 'https://www.fussball.de/spieltag/vl-grsued-hessen-verbandsliga-herren-saison2627-hessen/-/staffel/0318I2SMPG00000CVS5489BUVV628VP4-G#!/section/matches',
'hessenliga': 'https://www.fussball.de/spieltag/lotto-hessenliga-hessen-hessenliga-herren-saison2627-hessen/-/staffel/0318HOM0R000000BVS5489BUVV628VP4-G#!/section/matches',
'regionalliga_suedwest': 'https://www.fussball.de/spieltag/regionalliga-suedwest-deutschland-regionalliga-suedwest-herren-saison2627-deutschland/-/staffel/031D0NRQ3O000004VS5489BUVUR5FS5A-G#!/section/matches',
'3_bundesliga': 'https://www.fussball.de/spieltag/3liga-deutschland-3liga-herren-saison2627-deutschland/-/staffel/031BH6P20G00000HVS5489BUVUR5FS5A-G#!/section/matches',
'2_bundesliga': 'https://www.fussball.de/spieltag/2bundesliga-deutschland-2bundesliga-herren-saison2627-deutschland/-/staffel/031BG5ESP800000TVS5489BUVUR5FS5A-G#!/section/matches',
'bundesliga': 'https://www.fussball.de/spieltag/bundesliga-deutschland-bundesliga-herren-saison2627-deutschland/-/staffel/031BFREJC800001BVS5489BUVUR5FS5A-G#!/section/matches'
}

# write scraped tables from URL
write_scraped_df(d=liga_dict)
bl_to_vl = concat_scrape(liga_dict)
bl_to_vl.to_csv('bl_to_vl.csv', index=False)
bl_to_vl.info()

<class 'pandas.DataFrame'>
Index: 143 entries, 0 to 17
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   pl.platz            0 non-null      float64
 1   pl.platz.1          143 non-null    float64
 2   mannschaft          143 non-null    str    
 3   sp.spiele           143 non-null    int64  
 4   g                   143 non-null    int64  
 5   u                   143 non-null    int64  
 6   v                   143 non-null    int64  
 7   torv.torverhältnis  143 non-null    str    
 8   tordifferenz        143 non-null    int64  
 9   pkt.punkte          143 non-null    int64  
 10  liga                143 non-null    str    
dtypes: float64(2), int64(6), str(3)
memory usage: 13.4 KB


In [3]:
bl_to_vl

,pl.platz,pl.platz.1,mannschaft,sp.spiele,g,u,v,torv.torverhältnis,tordifferenz,pkt.punkte,liga
0,NaN,1.0,SG Klei./​Hun./​Doh.,18,12,2,4,42 : 26,16,38,verbandsliga_nord
1,NaN,2.0,SV Steinbach,17,11,3,3,45 : 22,23,36,verbandsliga_nord
2,NaN,3.0,Tuspo Rengershausen,18,10,5,3,52 : 28,24,35,verbandsliga_nord
3,NaN,4.0,SV Weidenhausen,17,11,1,5,26 : 19,7,34,verbandsliga_nord
4,NaN,5.0,SG Bad Soden,18,11,1,6,47 : 22,25,33,verbandsliga_nord
5,NaN,6.0,FSV Wolfhagen,18,10,2,6,29 : 17,12,32,verbandsliga_nord
6,NaN,7.0,OSC Vellmar,18,9,2,7,44 : 38,6,29,verbandsliga_nord
7,NaN,8.0,TSV 1900 Wabern,18,7,5,6,30 : 25,5,26,verbandsliga_nord
8,NaN,9.0,SV Buchonia Flieden,18,6,6,6,34 : 33,1,24,verbandsliga_nord
9,NaN,10.0,FSV Dörnberg,18,7,3,8,24 : 37,-13,24,verbandsliga_nord


In [4]:
gdf_list = []
geocode_manually = []
for i, mannschaft in enumerate(bl_to_vl['mannschaft']):
    try:
        geocoded_mannschaft = ox.geocode_to_gdf(mannschaft)
        geocoded_mannschaft['mannschaft'] = mannschaft
        # geocoded_mannschaft['liga'] = bl_to_vl.iloc[i]['liga'] # add liga w/ merge
        gdf_list.append(geocoded_mannschaft) # add the club name and league to the record for merging
    except:
        print(mannschaft)
        geocode_manually.append(mannschaft)
        continue

print('Geocoded Teams',len(gdf_list))
print('Need Geocoding',len(bl_to_vl))

SG Klei./​Hun./​Doh.
FSV Wolfhagen
OSC Vellmar
FSV Dörnberg
SG Barockstadt Fulda-Lehnerz II
SG Calden/​Meimbressen
FC Britannia Eichenzell
Tuspo Grebenstein
SG Bronnzell
Lichtenauer FV
TSV Steinbach Haiger II
SG Walluf
FV Biebrich 02
FC Ederbergland
SF/​BG Marburg
SV Zeilsheim
Türkischer SV Wiesbaden
SSC Juno Burg
SG Obbornh./​Bellersh.
TSF Heuchelheim
TuRa Niederhöchstadt
SV Pars Neu-Isenburg
DJK Spfr. Bad Homburg
SV Bosnien/​Herzegowina Ffm
Spfrd. Seligenstadt
VfR Fehlheim
SG Bornheim/​GW Ffm
SV Rot-Weiss Walldorf II
Kickers Offenbach II
SG Rot-Weiss Ffm.
1. Hanauer FC
Spvgg.Neu-Isenburg
TSV Eintr. Stadtallendorf
Türk Gücü Friedberg
SV Darmstadt 98 U21
FC Bayern Alzenau
FC Turabdin-Babylon
FSV Fernwald
Hünfelder SV
SC 1960 Hanau
SV Rot-Weiss Walldorf
SV Hummetroth
KSV Hessen Kassel II
SGV Heilbronn-Freiberg
VfR Mannheim
SV Stuttgarter Kickers
Kickers Offenbach
TSV Steinbach Haiger
SC Freiburg II
Eintracht Frankfurt II
SSV Ulm 1846 Fußball
1. FSV Mainz 05 II
1. FC Kaiserslautern II
MS

In [30]:
manually_geocoded_df = pd.read_csv('new_manually_geocoded.csv')
manually_geocoded_df.set_geometry('geometry')

TypeError: Input must be valid geometry objects: POLYGON ((8.1255004 50.7882854, 8.1272911 50.7878884, 8.1281342 50.7877556, 8.1293571 50.7873911, 8.1305558 50.7871347, 8.1321727 50.7869436, 8.1332489 50.7868044, 8.1336421 50.7867014, 8.1336057 50.7866185, 8.1330553 50.7866039, 8.132184 50.7867676, 8.1316282 50.7867325, 8.1307368 50.7868434, 8.1301807 50.7869552, 8.1297993 50.7869597, 8.129154 50.7872731, 8.1286171 50.7871716, 8.1288127 50.7865414, 8.1284397 50.7865249, 8.1283368 50.7867807, 8.1278238 50.7864856, 8.1278401 50.785834, 8.1283097 50.7850069, 8.1295412 50.7841368, 8.1292957 50.7830882, 8.1299042 50.7825576, 8.1311901 50.7813097, 8.1301508 50.7807634, 8.1308164 50.7802547, 8.129806 50.7794347, 8.1314043 50.7787128, 8.132189 50.7782693, 8.1313469 50.7768899, 8.1323493 50.7766858, 8.1328971 50.7765784, 8.1334496 50.7764771, 8.1349143 50.7757959, 8.1361092 50.7753402, 8.1364899 50.7749188, 8.1373902 50.7742971, 8.1381769 50.772745, 8.1392 50.7725165, 8.1405576 50.7722996, 8.14

In [ ]:

gdf = pd.concat(gdf_list)
gdf = pd.concat([gdf, manually_geocoded_df])
gdf = gdf.to_crs(CRS)
gdf.head(1)

AttributeError: 'DataFrame' object has no attribute 'to_crs'

In [ ]:
gdf['geometry'] = gdf['geometry'].centroid
gdf.info()

In [ ]:
gdf.explore(tooltip=False, popup=True)

In [ ]:
# write geocoded df
# gdf.to_file('data/hessen_fussball_gdf.geojson', driver='GeoJSON')